In [ ]:
# =============================================================================
# Created By:     Kai Metzger
# Created School: Franz-Oberthuer-Schule Wuerzburg
# Created Email:  metzgerkai@franz-oberthuer-schule.de
# Created Date:   Sat April 05 09:54 UTC 2025
# Changed Date:   Wed November 12 09:41 UTC 2026
# Version:        1.0.2
# =============================================================================
"""The Module has been build for training the symbols dataset with images + 
ground truth on a Raspberry Pi 5 with a standard USB camera. An image with a 
resolution of 640px x 480px can be recorded with the Python script 
create-symbols-dataset.py.
The files in the dataset <symbols> should be ordered in the following
manner:             > explanation
- symbols      
  - dataset1        > dataset version (dataset1 - 3 where used in the project)
    - data          > images (.png files)
    - gt            > ground truth (.txt files with class labels 1-4)
"""

# =============================================================================
# Import
# =============================================================================
import cv2
import os
import numpy as np
from keras import layers, models, utils, callbacks
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Enable Nvidia GPUs by un-commenting this line 
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
XLA_FLAGS="--xla_gpu_cuda_data_dir=/usr/"

# To get reproducable results with the same training setting random seed
SEED = 46
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [ ]:
# =============================================================================
# Declare variables
# =============================================================================
img_size_x = 32
img_size_y = 32
img_dim = img_size_x * img_size_y

# Change the following paths to your dataset path
home_dir = os.path.expanduser("~")
dataset_for_training = "dataset1"
img_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/data"
gt_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/gt"
checkpoint_filepath = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/chpt/"

print(img_dir)

In [ ]:
# =============================================================================
# Get number of data (image/label) 
# =============================================================================
anz_data = len(os.listdir(img_dir))
anz_data = int(anz_data)
dataset = np.zeros((anz_data, img_size_x, img_size_y, 3), dtype=float)
ground_truth = np.zeros((anz_data), dtype=int)
print("dataset size:", anz_data)

In [ ]:
# =============================================================================
# Define functions
# =============================================================================
# Change image size
def pic_prep(image, x, y):
    image = cv2.resize(image, (y, x))  # OpenCV expects (width, height)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image.astype(np.float32) / 255.0

In [ ]:
# =============================================================================
# Callback for training
# =============================================================================
# Set verbose flag to 0 to omit loss/accuracy output per epoch
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath + "chpt.keras",
    save_weights_only=False,
    monitor='val_accuracy',
    mode='auto',
    save_best_only=True,
    verbose = 1)

In [ ]:
# =============================================================================
# Read dataset
# =============================================================================
for i in range(0, anz_data, 1):
    img_name = img_dir + '/' + str(i) + '.png'      # create file names
    image = cv2.imread(img_name)                    # read image
    image = pic_prep(image, img_size_x, img_size_y) # prepare image
    dataset[i] = image                          # 2d-image to 3d-array
    
    txt_name = gt_dir + '/' + str(i) + '.txt'       # create gt file names
    ground_truth[i] = np.genfromtxt(txt_name, delimiter=',') 

In [ ]:
# =============================================================================
# Prepare dataset: train set (70%), and val/test set (30%)
# =============================================================================
# dataset has shape (number_of_images, img_dim)
print("dataset shape:", dataset.shape)
labels = ground_truth.reshape(-1) # integer labels 0–3
print("ground_truth shape:", ground_truth.shape)

# 70% training, 30% temporary
train_data, temp_data, train_labels, temp_labels = train_test_split(
    dataset,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels,
)

# Split the temporary 30% equally: 15% validation, 15% test
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels,
)

# Convert integer labels to one-hot labels for categorical_crossentropy
train_gt = utils.to_categorical(train_labels, num_classes=4)
val_gt = utils.to_categorical(val_labels, num_classes=4)
test_gt = utils.to_categorical(test_labels, num_classes=4)

print("Train:", len(train_data), "Validation:", len(val_data), "Test:", len(test_data))

print("Labels and counts:", np.unique(ground_truth, return_counts=True))
print("Data:", train_data.shape, train_data.min(), train_data.max())
print("One-hot labels:", train_gt.shape)

In [ ]:
# 1. Confirm labels and image values are valid:
labels = ground_truth.reshape(-1)

print("Image shape:", dataset.shape)
print("Pixel range:", dataset.min(), "to", dataset.max())
print("Labels/counts:", np.unique(labels, return_counts=True))

assert dataset.shape == (len(labels), 32, 32, 3)
assert np.isfinite(dataset).all()
assert set(np.unique(labels)).issubset({0, 1, 2, 3})

# 2. Verify labels still match their files. Since your filenames use numeric IDs, compare a few loaded labels against the corresponding ground-truth files:
for i in [0, 1, 2, 3, 10, 25]:
    file_label = int(np.genfromtxt(f"{gt_dir}/{i}.txt", delimiter=","))
    print(i, "loaded:", labels[i], "file:", file_label)

# Each pair should match. Also check that all expected image and label files exist:
missing = [
    i for i in range(anz_data)
    if not os.path.exists(f"{img_dir}/{i}.png")
    or not os.path.exists(f"{gt_dir}/{i}.txt")
]
print("Missing file IDs:", missing)

# 3. Look at labeled images. This often catches incorrect labels or unexpected preprocessing immediately:
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(dataset[i])
    ax.set_title(f"label={labels[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# 4. Check class counts after splitting:
for name, one_hot_labels in [
    ("train", train_gt),
    ("validation", val_gt),
    ("test", test_gt),
]:
    class_ids = np.argmax(one_hot_labels, axis=1)
    print(name, np.unique(class_ids, return_counts=True))

In [ ]:
# =============================================================================
# Create neural network with 4 layers and (64, 32, 16, 4) neurons per layer.
# =============================================================================
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Lambda(tf.image.rgb_to_grayscale),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    #layers.Dropout(0.3),    
    layers.Dense(192, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(4, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print(model.summary())


In [ ]:
# =============================================================================
# Train the neuronal network e.g. for 300 epochs.
# =============================================================================
history = model.fit(
    train_data,
    train_gt,
    batch_size=8,
    epochs=100,
    verbose=1,
    shuffle=True,
    validation_data=(val_data, val_gt),
    callbacks=[model_checkpoint_callback],
)

In [ ]:
# =============================================================================
# Plot train and val accuracy.
# =============================================================================

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# =============================================================================
# Plot train and val loss.
# =============================================================================

plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='lower left')
plt.show()

In [ ]:
# =============================================================================
# Load pretrained dataset weights to e.g. test on new (unseen) data.
# =============================================================================
model.load_weights(checkpoint_filepath + "chpt.keras")

In [ ]:
# =============================================================================
# Test dataset on xxx.
# =============================================================================
score = model.evaluate(test_data, test_gt, verbose=1)
print('Test score:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# =============================================================================
# Testing on a single image.
# =============================================================================
img_path = (
    home_dir + "/ml_project/datasets/symbols/"
    + dataset_for_training + "/data/16.png"
)

img_pred = cv2.imread(img_path)
if img_pred is None:
    raise FileNotFoundError(f"Could not read image: {img_path}")

img_pred = pic_prep(img_pred, img_size_x, img_size_y)
data_pred = np.expand_dims(img_pred, axis=0)  # (1, 32, 32, 3)

result = model.predict(data_pred, verbose=0)

print("Probability for classes: (cross, circle, triangle, square) in percent", 
      result)

In [ ]:
# =============================================================================
# Output class: 
# translate class label (0,1,2,3) to class (cross, circle, triangle, square).
# =============================================================================

max_res = 0
res_index = 4
for i in range(0, 4, 1):
    if result[0,i] > max_res:
        max_res = result[0,i]
        res_index = i

if res_index == 0:
    print('Cross detected!')
elif res_index == 1:
    print('Circle detected!')
elif res_index == 2:
    print('Triangle detected!')
elif res_index == 3:
    print('Square detected!')
elif res_index == 4:
    print('Error!')